# torch.stack() 详解
## 核心定义
torch.stack() 的作用：沿着一个新维度，将一组张量"堆叠"起来。

In [ ]:
torch.stack(tensors, dim=0)

| 参数        | 说明                                  |
| --------- | ----------------------------------- |
| `tensors` | 张量序列（list/tuple），**所有张量形状必须完全相同**   |
| `dim`     | 新插入的维度索引，范围是 `[-(N+1), N+1]`，默认 `0` |


### 关键特征：
- <font color='red'>不修改原有张量的形状，而是新增一个维度</font>
- 所有输入张量必须形状完全一致
- 结果张量的 shape = [len(tensors)] + 单个张量的 shape

# 与 torch.cat() 的本质区别

| 操作      | 作用          | 维度变化   | 形状要求           |
| ------- | ----------- | ------ | -------------- |
| `stack` | **新增维度**堆叠  | 维度数 +1 | 所有张量形状**完全相同** |
| `cat`   | **沿现有维度**拼接 | 维度数不变  | 除拼接维度外，其余维度相同  |

## 对比示例

In [3]:
import torch

a = torch.tensor([1,2,3]) # shape [3]
b = torch.tensor([4,5,6]) # shape [3]

# stack: 新增一个维度
stacked = torch.stack([a,b],dim=0)
print(stacked)
print(stacked.shape)

# cat: 沿着已有维度拼接
concat = torch.concat([a,b],dim=0)
print(concat)
print(concat.shape)

tensor([[1, 2, 3],
        [4, 5, 6]])
torch.Size([2, 3])
tensor([1, 2, 3, 4, 5, 6])
torch.Size([6])


# 参数 dim 详解
dim 控制新维度插入的位置。

## 示例：3个形状为 [2, 3] 的张量

In [9]:
a = torch.ones(2, 3)   # [[1,1,1], [1,1,1]]
b = torch.ones(2, 3) * 2
print(b)
c = torch.ones(2, 3) * 3
print(c)

tensor([[2., 2., 2.],
        [2., 2., 2.]])
tensor([[3., 3., 3.],
        [3., 3., 3.]])


| `dim`        | 结果形状        | 直观理解              |
| ------------ | ----------- | ----------------- |
| `dim=0` (默认) | `[3, 2, 3]` | 3个"层"，每层是 `[2,3]` |
| `dim=1`      | `[2, 3, 3]` | 每行变成3个"列块"        |
| `dim=2`      | `[2, 3, 3]` | 每个元素位置堆叠3个值       |
| `dim=-1`     | `[2, 3, 3]` | 同 `dim=2`（最后一个维度） |


## 可视化 dim 的不同取值

In [11]:
# dim=0: [3, 2, 3]
# 想象成3张纸叠在一起
result_0 = torch.stack([a, b, c], dim=0)
print(f"result_0:\n{result_0}")

# dim=1: [2, 3, 3]
# 想象成每行内，把3个张量的对应行并排
result_1 = torch.stack([a, b, c], dim=1)
print(f"result_1:\n{result_1}")

# dim=2: [2, 3, 3]
# 想象成每个位置，把3个张量的对应值堆成一列
result_2 = torch.stack([a, b, c], dim=2)
print(f"result_2:\n{result_2}")

result_0:
tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[2., 2., 2.],
         [2., 2., 2.]],

        [[3., 3., 3.],
         [3., 3., 3.]]])
result_1:
tensor([[[1., 1., 1.],
         [2., 2., 2.],
         [3., 3., 3.]],

        [[1., 1., 1.],
         [2., 2., 2.],
         [3., 3., 3.]]])
result_2:
tensor([[[1., 2., 3.],
         [1., 2., 3.],
         [1., 2., 3.]],

        [[1., 2., 3.],
         [1., 2., 3.],
         [1., 2., 3.]]])


# 完整示例
## 示例1：基础用法（构建批次）

In [12]:
import torch

# 单张图片: [3, 224, 224] (通道, 高, 宽)
img1 = torch.randn(3, 224, 224)
img2 = torch.randn(3, 224, 224)
img3 = torch.randn(3, 224, 224)

# 堆叠成批次: [batch_size, 通道, 高, 宽]
batch = torch.stack([img1, img2, img3], dim=0)
print(batch.shape)  # torch.Size([3, 3, 224, 224])

torch.Size([3, 3, 224, 224])


## 示例2：不同 dim 的效果


In [2]:
import torch
a = torch.tensor([[1, 2], [3, 4]])  # [2, 2]
b = torch.tensor([[5, 6], [7, 8]])  # [2, 2]

# dim=0
r0 = torch.stack([a, b], dim=0)
print(r0.shape)   # [2, 2, 2]
print(r0)
# tensor([[[1, 2],
#          [3, 4]],
#         [[5, 6],
#          [7, 8]]])

# dim=1
r1 = torch.stack([a, b], dim=1)
print(r1.shape)   # [2, 2, 2]
print(r1)
# tensor([[[1, 2],
#          [5, 6]],
#         [[3, 4],
#          [7, 8]]])

# dim=2
r2 = torch.stack([a, b], dim=2)
print(r2.shape)   # [2, 2, 2]
print(r2)
# tensor([[[1, 5],
#          [2, 6]],
#         [[3, 7],
#          [4, 8]]])

torch.Size([2, 2, 2])
tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])
torch.Size([2, 2, 2])
tensor([[[1, 2],
         [5, 6]],

        [[3, 4],
         [7, 8]]])
torch.Size([2, 2, 2])
tensor([[[1, 5],
         [2, 6]],

        [[3, 7],
         [4, 8]]])


## 示例3：负维度索引

In [4]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])
print(a.shape)

# dim=-1 等价于 dim=1（最后一个维度）
r = torch.stack([a, b], dim=-1)
print(r)
# tensor([[1, 4],
#         [2, 5],
#         [3, 6]])
# shape: [3, 2]

torch.Size([3])
tensor([[1, 4],
        [2, 5],
        [3, 6]])


# 常见应用场景
## 场景1：将列表转为批次张量

In [ ]:
# 从 DataLoader 得到的是 list of tensors
samples = [torch.randn(3, 32, 32) for _ in range(8)]  # 8张图

# 堆叠成 [8, 3, 32, 32]
batch = torch.stack(samples, dim=0)

## 场景2：构造序列数据（RNN/Transformer）

In [ ]:
# 每个时间步的词向量
word_vecs = [torch.randn(128) for _ in range(10)]  # 10个词

# 堆叠成 [seq_len, hidden_dim]
sequence = torch.stack(word_vecs, dim=0)  # [10, 128]

## 场景3：收集多分支输出

In [ ]:
# 多尺度特征图
feat1 = torch.randn(64, 28, 28)
feat2 = torch.randn(64, 28, 28)
feat3 = torch.randn(64, 28, 28)

# 堆叠后方便后续处理
multi_scale = torch.stack([feat1, feat2, feat3], dim=0)  # [3, 64, 28, 28]

# 常见错误
## ❌ 错误1：形状不一致
- 解决：<font color='red'>先用 unsqueeze 统一形状，或检查数据。</font>

In [5]:
a = torch.tensor([1, 2, 3]).unsqueeze(0)      # [3]
b = torch.tensor([[1, 2, 3]])    # [1, 3]

torch.stack([a, b])  
# RuntimeError: stack expects each tensor to be equal size...

tensor([[[1, 2, 3]],

        [[1, 2, 3]]])

## ❌ 错误2：dim 超出范围

In [ ]:
a = torch.tensor([1, 2, 3])  # 1维张量，有效dim范围: [-2, 1]

torch.stack([a, a], dim=2)  
# RuntimeError: dim=2 out of range [-2, 1]

## ❌ 错误3：与 cat 混淆导致维度错误

In [ ]:
# 错误：用 cat 期望得到 batch 维度
imgs = [torch.randn(3, 32, 32) for _ in range(4)]
batch = torch.cat(imgs, dim=0)  # 结果: [12, 32, 32] ❌

# 正确：用 stack
batch = torch.stack(imgs, dim=0)  # 结果: [4, 3, 32, 32] ✓

# 快速记忆口诀
- cat 是"拼接"（**变长**），
- stack 是"堆叠"（**变厚**）

cat 沿已有维度走，stack 新开一个维度放。